# 05 Advanced Analytics
Combine VaR review, investor cohorts, and an explainable fund recommender.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd(); DATA = ROOT / 'data' / 'processed'
transactions = pd.read_csv(DATA / 'cleaned_08_investor_transactions.csv', parse_dates=['transaction_date'])
cohorts = transactions.assign(cohort_month=transactions.transaction_date.dt.to_period('M').astype(str)).groupby(['age_group', 'city_tier'], as_index=False).agg(investors=('investor_id', 'nunique'), transactions=('investor_id', 'size'), avg_amount_inr=('amount_inr', 'mean'), total_amount_inr=('amount_inr', 'sum')).sort_values('total_amount_inr', ascending=False)
cohorts.head(10)

In [ ]:
performance = pd.read_csv(DATA / 'cleaned_07_scheme_performance.csv')
performance['recommendation_score'] = performance['sharpe_ratio'].rank(pct=True) * 0.5 + performance['return_1yr_pct'].rank(pct=True) * 0.3 + (1 - performance['std_dev_ann_pct'].rank(pct=True)) * 0.2
performance.sort_values('recommendation_score', ascending=False)[['scheme_name', 'category', 'sharpe_ratio', 'std_dev_ann_pct', 'recommendation_score']].head(10)

## Interpretation
The recommender is a ranking aid, not financial advice. It favors risk-adjusted return, then one-year return, while penalizing volatility. Validate suitability against investor objectives and risk tolerance.